# How to read a Parquet file with Python

Load Parquet into pandas with Rugo, read only the columns and rows you need, look inside a file before reading it, and query Parquet with SQL using Opteryx.

This notebook accompanies [How to read a Parquet file with Python](https://docs.opteryx.app/docs/guides/read-a-parquet-file-with-python) on docs.opteryx.app.

In [ ]:
%pip install -q rugo opteryx-core pandas pyarrow

## Get the data

`space_missions.parquet`: 4,630 orbital launch attempts since Sputnik.

In [ ]:
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/mabel-dev/opteryx-core/main/testdata/flat/space_missions/space_missions.parquet", "space_missions.parquet")

## Load a Parquet file into pandas

`read_parquet` yields one morsel per row group. Convert each to Arrow, concatenate, and convert once to pandas.

In [ ]:
import pyarrow
from rugo import parquet

with parquet.read_parquet("space_missions.parquet") as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

df.dtypes

In [ ]:
df.head()

## Read only the columns you need

A column you don't ask for is never read from disk or decompressed.

In [ ]:
with parquet.read_parquet("space_missions.parquet", columns=["Company", "Mission", "Price"]) as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

df.head()

## Filter while reading

`predicates` is a list of `(column, operator, value)` tuples, combined with AND.

In [ ]:
with parquet.read_parquet(
    "space_missions.parquet",
    columns=["Company", "Mission", "Price"],
    predicates=[("Company", "=", "SpaceX"), ("Price", "<", 60)],
) as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

len(df)

In [ ]:
# `in`, `!=`, and a predicate column that isn't in `columns`
with parquet.read_parquet(
    "space_missions.parquet",
    columns=["Company", "Mission"],
    predicates=[("Company", "in", ["SpaceX", "Rocket Lab"]), ("Mission_Status", "!=", "Success")],
) as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

df

In [ ]:
import datetime

with parquet.read_parquet(
    "space_missions.parquet",
    columns=["Mission"],
    predicates=[("Lauched_at", ">=", datetime.datetime(2020, 1, 1))],
) as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

len(df)

## Look inside a file first

The `rugo` command reads the footer — schema, row counts, statistics — without reading the data.

In [ ]:
!rugo info space_missions.parquet

In [ ]:
!rugo schema space_missions.parquet

In [ ]:
!rugo inspect space_missions.parquet

In [ ]:
meta = parquet.read_metadata("space_missions.parquet")
print(meta.num_rows, "rows in", meta.num_row_groups, "row group(s)")
for column in meta.schema_columns:
    print(column.name, column.logical_type)

## Files larger than memory

Work through a file one morsel (one row group) at a time, keeping only what you need.

In [ ]:
totals = {}
with parquet.read_parquet("space_missions.parquet", columns=["Company", "Price"]) as reader:
    for morsel in reader:
        chunk = morsel.to_arrow().to_pandas()
        for company, price in chunk.groupby("Company")["Price"].sum().items():
            totals[company] = totals.get(company, 0) + price

sorted(totals.items(), key=lambda kv: -kv[1])[:5]

## Query with SQL: Opteryx

Aggregate inside the engine and only bring the answer into pandas.

In [ ]:
import opteryx

morsels = opteryx.session().execute_to_morsels("""
    SELECT Company,
           COUNT(*)             AS launches,
           ROUND(AVG(Price), 1) AS avg_price_musd
      FROM READ_PARQUET('space_missions.parquet')
     WHERE Mission_Status = 'Success'
       AND Lauched_at >= CAST('2000-01-01' AS TIMESTAMP)
     GROUP BY Company
     ORDER BY launches DESC
     LIMIT 5
""")
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()

In [ ]:
# EXPLAIN shows the filter pushed into the read
morsels = opteryx.session().execute_to_morsels(
    "EXPLAIN SELECT Mission FROM READ_PARQUET('space_missions.parquet') WHERE Company = 'SpaceX'"
)
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()

In [ ]:
# bind values as parameters rather than formatting them into the SQL
morsels = opteryx.session().execute_to_morsels(
    "SELECT Mission, Price FROM READ_PARQUET('space_missions.parquet') WHERE Company = :company AND Price < :max_price",
    params={"company": "SpaceX", "max_price": 60},
)
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas().head()